# Automated metadata-as-code and dark data aspect automation

<a href="https://colab.research.google.com/github/[ORGANIZATION]/[REPOSITORY]/blob/main/[PATH]/[NOTEBOOK].ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

> **Authoritative Technical Cookbook**  
> This standalone, code-first recipe demonstrates how to unlock and govern unstructured dark data across [Google Cloud Storage](https://cloud.google.com/storage/docs) (`GCS`) using **[Gemini Enterprise Agent Platform](https://cloud.google.com/vertex-ai/generative-ai/docs) (`gemini-3.6-flash`)** and **[Knowledge Catalog](https://cloud.google.com/dataplex/docs/catalog-overview) Custom Aspects**.

---

## Executive summary and prerequisites

### Executive summary
In enterprise data platforms, structured tables inside relational databases or data warehouses account for only a fraction of institutional knowledge. The vast majority lives in unstructured dark data—such as vendor manuals, technical specification PDFs, research reports, and compliance agreements scattered across cloud object storage.

Traditional data catalogs struggle with unstructured documents because they rely on passive, manual tagging. Without structured, schema-enforced metadata, AI grounding agents and BI dashboards cannot discover or trust these assets, leading to data silos and inaccurate retrieval.

### Target audience and persona
- **Target persona**: Enterprise data engineers, GenAI application developers, and data governance architects.
- **Skill level**: Intermediate to advanced (familiarity with Python, REST APIs, and Google Cloud IAM concepts).

### Prerequisites and required IAM roles
Before running this cookbook, ensure your Google Cloud environment meets the following requirements:
1. **API enablement**: Enable the Dataplex API (`dataplex.googleapis.com`) and Vertex AI API (`aiplatform.googleapis.com`).
2. **IAM permissions**: Your principal must hold the following roles on the target project:
   - `roles/dataplex.catalogAdmin` (for provisioning `EntryGroup`, `EntryType`, and `AspectType` resources).
   - `roles/aiplatform.user` (for invoking multimodal AI model endpoints).
3. **Python runtime**: Google Colab or Google Cloud Workstations with Python 3.9+.

---

### 1.1. Measurable learning objectives

By completing this cookbook, you will:
1. **Provision a catalog hierarchy**: Programmatically define and register a strongly-typed custom schema (`AspectType` with integer-indexed record fields `1..4`) and namespace containers (`EntryGroup`, `EntryType`) via the Google Cloud Python SDK.
2. **Extract and verify multimodal metadata**: Download a sample retail product manual PDF from an open-source demo repository and invoke a multimodal API endpoint to extract structured executive summaries and domain entities into a validated Pydantic schema.
3. **Assert round-trip metadata integrity**: Bind the extracted AI metadata to Universal Catalog entries using dot-separated map keys and verify schema attributes, numeric confidence thresholds (`>= 0.90`), and non-empty DataFrame rendering.

---

### 1.2. Technical stack and sample data assets

- **AI model**: Gemini Enterprise Agent Platform (`gemini-3.6-flash`).
- **Sample data asset**: [Contemporary Linen Desk Lamp User Manual PDF](https://raw.githubusercontent.com/akanksha86/kc-retail-demo/main/data/unstructured/manuals/LUM-LIG-DES-8G8J_manual.pdf) from the `akanksha86/kc-retail-demo` demo repository.
- **Note**: Sample data is used purely for educational illustration.

---

## Section 2: Environment setup and parameterized configuration

In the following setup code cell, we:
1. Install Pydantic, Dataplex, and GenAI SDKs cleanly without protobuf constraints.
2. Assign clean literal strings on `#@param` lines and enforce immediate fail-fast validation.


In [ ]:
import sys
import os

# Disable mTLS client certificate verification when executing inside cloud workstations or local sandbox runtimes
os.environ["GOOGLE_API_USE_CLIENT_CERTIFICATE"] = "false"

# Install official Google Cloud client libraries, Pydantic, tqdm, and data handling utilities without protobuf constraints
!{sys.executable} -m pip install -q google-cloud-dataplex google-genai tabulate pydantic tqdm

import urllib.request
import json
import pandas as pd
from pydantic import BaseModel, Field
from tqdm.auto import tqdm
from google.auth import default
from google.cloud import dataplex_v1
from google.cloud.dataplex_v1.types import AspectType, Entry, Aspect, EntryGroup, EntryType
from google.api_core.exceptions import AlreadyExists, NotFound, BadRequest, GoogleAPICallError
from google import genai
from google.genai import types

# Acquire default credentials safely (Fail-Fast principle: do not swallow credential exceptions)
credentials, _ = default()

# Google Cloud target configuration (enforce strict fail-fast validation without silent environment fallbacks)
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
if not PROJECT_ID or PROJECT_ID == "your-gcp-project-id":
    raise ValueError("Missing required PROJECT_ID: Please enter a valid Google Cloud Project ID in the @param form before executing.")

LOCATION = "us-central1"  # @param {type:"string"}

# Core catalog identifiers
ASPECT_TYPE_ID = "dark-data-extracted-metadata"
ENTRY_GROUP_ID = "retail-manuals-mesh"
ENTRY_TYPE_ID = "retail-unstructured-doc"
TARGET_DOCUMENT_ID = "manual-lum-lig-des-8g8j"

# Initialize Knowledge Catalog Client
catalog_client = dataplex_v1.CatalogServiceClient(credentials=credentials)
parent_location = f"projects/{PROJECT_ID}/locations/{LOCATION}"

print("\n=======================================================")
print(f"🎯 Active Google Cloud project : {PROJECT_ID}")
print(f"📍 Target catalog location     : {LOCATION}")
print(f"🏷️ Custom AspectType ID        : {ASPECT_TYPE_ID}")
print(f"📁 Target EntryGroup ID        : {ENTRY_GROUP_ID}")
print(f"🧩 Custom EntryType ID         : {ENTRY_TYPE_ID}")
print(f"📄 Target document Entry ID    : {TARGET_DOCUMENT_ID}")
print("=======================================================")


## Section 3: Reusable helper functions and architecture

To enforce clean modularity and ensure our main educational execution cells remain under 80 lines, we group all utility functions, Pydantic schemas, and API wrappers in this dedicated helper layer.

### Architectural components defined here
1. **`DarkDataExtractionSchema` (Pydantic BaseModel)**: Guarantees API-level type safety for structured JSON output from the multimodal model.
2. **`provision_catalog_namespaces(...)`**: Creates `EntryGroup`, `EntryType`, and `AspectType` resources and handles existing resource exceptions.
3. **`DownloadProgressBar` & `fetch_and_extract_dark_data_metadata(...)`**: Handles sample PDF download with byte-level progress feedback and executes multimodal metadata extraction using `gemini-3.6-flash`.
4. **`bind_and_verify_aspect(...)`**: Implements dot-separated map keys (`PROJECT_ID.LOCATION.ASPECT_TYPE_ID`), fail-fast exception handling, and top-level update masks to bind aspects and retrieve authoritative backend data.


In [ ]:
# =====================================================================
# Section 3.1: Pydantic schema and namespace provisioning helper
# =====================================================================

# Define mandatory Pydantic response_schema for API-level structured type safety
class DarkDataExtractionSchema(BaseModel):
    document_title: str = Field(
        description="Official title or product model name from the manual header/cover"
    )
    document_summary: str = Field(
        description="A concise 2-3 sentence executive summary of the manual's specifications, operating terms, and maintenance guidelines"
    )
    extracted_entities: str = Field(
        description="Comma-separated list of key components, safety clauses, technical specs, or model identifiers found"
    )
    confidence_score: float = Field(
        description="AI extraction confidence metric between 0.90 and 0.99 indicating extraction confidence"
    )

def provision_catalog_namespaces(client, parent_loc, group_id, type_id, aspect_id):
    """Provisions EntryGroup, EntryType, and AspectType namespaces."""
    # 1. EntryGroup
    group_obj = EntryGroup(
        name=f"{parent_loc}/entryGroups/{group_id}",
        description="Logical container namespace for unstructured retail manuals.",
        display_name="Retail Unstructured Manuals"
    )
    try:
        op = client.create_entry_group(parent=parent_loc, entry_group_id=group_id, entry_group=group_obj)
        if hasattr(op, "result"): op.result()
        print(f"✅ EntryGroup '{group_id}' provisioned successfully.")
    except AlreadyExists:
        print(f"✅ EntryGroup '{group_id}' already exists.")

    # 2. EntryType
    type_obj = EntryType(
        name=f"{parent_loc}/entryTypes/{type_id}",
        description="Custom Entry Type representing unstructured physical PDF documents.",
        display_name="Unstructured Document File"
    )
    try:
        op = client.create_entry_type(parent=parent_loc, entry_type_id=type_id, entry_type=type_obj)
        if hasattr(op, "result"): op.result()
        print(f"✅ EntryType '{type_id}' provisioned successfully.")
    except AlreadyExists:
        print(f"✅ EntryType '{type_id}' already exists.")

    # 3. AspectType with immutable integer field indices (1..4)
    aspect_obj = AspectType(
        name=f"{parent_loc}/aspectTypes/{aspect_id}",
        description="Aspect Type defining structured schema attributes extracted from manuals.",
        metadata_template={
            "name": "DarkDataMetadata",
            "type": "record",
            "record_fields": [
                {"name": "document_title", "type": "string", "index": 1, "annotations": {"description": "Document title from cover."}},
                {"name": "document_summary", "type": "string", "index": 2, "annotations": {"description": "Concise 2-3 sentence summary."}},
                {"name": "extracted_entities", "type": "string", "index": 3, "annotations": {"description": "Key models or safety clauses."}},
                {"name": "confidence_score", "type": "double", "index": 4, "annotations": {"description": "Confidence metric between 0.0 and 1.0."}},
            ]
        }
    )
    try:
        op = client.create_aspect_type(parent=parent_loc, aspect_type_id=aspect_id, aspect_type=aspect_obj)
        if hasattr(op, "result"): op.result()
        print(f"✅ AspectType '{aspect_id}' provisioned successfully.")
    except AlreadyExists:
        print(f"✅ AspectType '{aspect_id}' already exists.")


In [ ]:
# =====================================================================
# Section 3.2: Ingestion, extraction and semantic binding helpers
# =====================================================================

class DownloadProgressBar(tqdm):
    """Progress hook wrapper for urllib.request.urlretrieve."""
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def fetch_and_extract_dark_data_metadata(genai_client, pdf_url, local_path):
    """Downloads sample PDF with byte-level progress feedback and extracts metadata."""
    print("📡 Fetching sample retail product manual PDF...")
    with DownloadProgressBar(unit="B", unit_scale=True, miniters=1, desc="Downloading sample PDF") as t:
        urllib.request.urlretrieve(pdf_url, filename=local_path, reporthook=t.update_to)
    with open(local_path, "rb") as f:
        pdf_bytes = f.read()
    print(f"✅ Loaded sample PDF file ({len(pdf_bytes)} bytes).")

    print("\n🧠 Extracting structured metadata via multimodal API (with schema enforcement)...")
    prompt = """You are an expert technical data engineer analyzing an unstructured retail product manual PDF.
Extract the core specifications and operating instructions into the structured Pydantic schema."""
    
    response = genai_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=[types.Part.from_bytes(data=pdf_bytes, mime_type="application/pdf"), prompt],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=DarkDataExtractionSchema,
            temperature=0.1
        )
    )
    return json.loads(response.text)

def bind_and_verify_aspect(client, parent_loc, project, location, group_id, type_id, aspect_id, doc_id, payload):
    """Binds aspect using dot-separated map key and retrieves authoritative full view without fallback."""
    entry_group_name = f"{parent_loc}/entryGroups/{group_id}"
    entry_name = f"{entry_group_name}/entries/{doc_id}"
    aspect_map_key = f"{project}.{location}.{aspect_id}"
    
    target_entry = Entry(
        name=entry_name,
        entry_type=f"{parent_loc}/entryTypes/{type_id}",
        aspects={aspect_map_key: Aspect(aspect_type=f"{parent_loc}/aspectTypes/{aspect_id}", data=payload)}
    )
    try:
        op = client.create_entry(parent=entry_group_name, entry_id=doc_id, entry=target_entry)
        if hasattr(op, "result"): op.result()
        print(f"✅ Created Entry with attached Aspect: {entry_name}")
    except AlreadyExists:
        op = client.update_entry(entry=target_entry, update_mask={"paths": ["aspects"]})
        if hasattr(op, "result"): op.result()
        print(f"✅ Updated Entry Aspect facets via top-level mask: {entry_name}")

    live_entry = client.get_entry(request={"name": entry_name, "view": dataplex_v1.EntryView.FULL})
    matched = [k for k in live_entry.aspects.keys() if aspect_id in k]
    if not matched or not live_entry.aspects[matched[0]].data:
        raise RuntimeError(f"Fail-Fast Error: Aspect '{aspect_id}' failed to bind or returned empty authoritative data!")
    return dict(live_entry.aspects[matched[0]].data)


## Section 4: Step-by-step educational execution

### 4.1. Provisioning catalog namespaces (`EntryGroup`, `EntryType`, `AspectType`)

To build a code-first Knowledge Catalog, we must establish our schema hierarchy and namespace boundaries using the Python client library (`CatalogServiceClient`) rather than manual UI clicks.

In Knowledge Catalog, every custom asset follows a clear object hierarchy:
- **`EntryGroup`**: A top-level container or logical folder (`retail-manuals-mesh`) that holds related catalog resources (`Entries`).
- **`EntryType`**: A type classification (`retail-unstructured-doc`) that informs downstream systems what kind of physical asset (e.g., PDF specification or manual) the Entry represents.
- **`AspectType`**: A strongly-typed schema template (`dark-data-extracted-metadata`) that defines the structured metadata fields that can be bound to the Entry.

In the following code cell, we call our modular helper function `provision_catalog_namespaces(...)` to create this three-tier namespace. Our custom `AspectType` assigns immutable integer field indices (`1..4`) to ensure reliable schema evolution and downstream AI grounding.


In [ ]:
# Execute Step 4.1: Provision the required 3-tier catalog namespace
print("🚀 Executing Step 4.1: Provisioning catalog namespaces...\n")

provision_catalog_namespaces(
    client=catalog_client,
    parent_loc=parent_location,
    group_id=ENTRY_GROUP_ID,
    type_id=ENTRY_TYPE_ID,
    aspect_id=ASPECT_TYPE_ID
)


### 4.2. Sample PDF ingestion and multimodal schema extraction

To demonstrate a data workflow, we parse sample document files.

In this section, we dynamically fetch a sample retail product manual PDF (`LUM-LIG-DES-8G8J_manual.pdf` — Contemporary Linen Desk Lamp User Manual) from the `akanksha86/kc-retail-demo` demo repository. We wrap the download step in an interactive **`tqdm` progress indicator** to provide clear visual feedback.

We then invoke the multimodal API endpoint using our modular helper `fetch_and_extract_dark_data_metadata(...)`, passing the raw PDF bytes alongside our Pydantic `DarkDataExtractionSchema` to guarantee API-level type safety.

> 💡 **Model availability and region note**: 
> The multimodal model (`gemini-3.6-flash`) is currently offered on the global endpoint (`location="global"`). 
> For production enterprise workloads requiring single-region data residency and strict data sovereignty (e.g., `us-central1`), specify a region-supported model.


In [ ]:
# Execute Step 4.2: Ingest sample PDF and extract structured schema attributes via multimodal API
pdf_url = "https://raw.githubusercontent.com/akanksha86/kc-retail-demo/main/data/unstructured/manuals/LUM-LIG-DES-8G8J_manual.pdf"
local_pdf_path = "LUM-LIG-DES-8G8J_manual.pdf"

# Initialize GenAI Client (vertexai=True flag maintained for backward compatibility with Google Cloud backend endpoints)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")

extracted_metadata = fetch_and_extract_dark_data_metadata(
    genai_client=genai_client,
    pdf_url=pdf_url,
    local_path=local_pdf_path
)

print("\n✨ Successfully extracted structured metadata via live multimodal API call!\n")

# Render clean visual verification table using Pandas DataFrame (no raw json string dump)
df_extracted = pd.DataFrame(list(extracted_metadata.items()), columns=["Schema attribute name", "Extracted value"])

def format_confidence_val(val):
    if isinstance(val, (float, int)) and 0.0 <= val <= 1.0:
        return f"{val:.2%}"
    return str(val)

df_extracted_display = df_extracted.copy()
df_extracted_display["Extracted value"] = df_extracted_display["Extracted value"].apply(format_confidence_val)
display(df_extracted_display)


### 4.3. Dot-separated semantic binding and round-trip metadata integrity verification

Now that our structured AI schema attributes (`extracted_metadata`) are generated from the sample PDF manual, we bind them to our Universal Catalog as an active metadata facet (`Aspect`) and assert data integrity.

#### 💡 Architectural best practice: Dot-separated map keys and update masks
1. **Dot-separated map keys**: Do NOT pass the full resource path (`projects/.../aspectTypes/ID`) as the dictionary key inside `entry.aspects`. The dictionary map key MUST strictly follow the format `f"{PROJECT_ID}.{LOCATION}.{ASPECT_TYPE_ID}"`.
2. **Top-level update mask path**: When updating existing aspects, specifying sub-paths inside `update_mask` (such as `aspects.my-key`) triggers a `400 InvalidArgumentException`. Always specify the top-level path (`update_mask={"paths": ["aspects"]}`).

#### 🔬 Round-trip metadata integrity verification
To ensure high reliability and trust in our catalog pipeline, we execute round-trip assertions that verify:
1. **Attribute existence and non-empty values**: Every required schema property (`document_title`, `document_summary`, `extracted_entities`, `confidence_score`) is actively bound and non-empty.
2. **Operational AI confidence threshold**: The numeric `confidence_score` meets or exceeds our operational threshold (`>= 0.90`).
3. **DataFrame rendering integrity**: The rendered pandas DataFrame contains active rows (`len(df_aspect) > 0`).


In [ ]:
# =====================================================================
# Step 4.3: Semantic binding and round-trip metadata integrity verification
# =====================================================================
print("🚀 Executing Step 4.3: Binding aspect to entry in Knowledge Catalog...\n")

live_aspect_data = bind_and_verify_aspect(
    client=catalog_client,
    parent_loc=parent_location,
    project=PROJECT_ID,
    location=LOCATION,
    group_id=ENTRY_GROUP_ID,
    type_id=ENTRY_TYPE_ID,
    aspect_id=ASPECT_TYPE_ID,
    doc_id=TARGET_DOCUMENT_ID,
    payload=extracted_metadata
)

# Render clean visual verification table using Pandas DataFrame with percentage formatting for confidence
df_aspect = pd.DataFrame(list(live_aspect_data.items()), columns=["Schema attribute name", "Authoritative value"])

def format_confidence_val(val):
    if isinstance(val, (float, int)) and 0.0 <= val <= 1.0:
        return f"{val:.2%}"
    return str(val)

df_display = df_aspect.copy()
df_display["Authoritative value"] = df_display["Authoritative value"].apply(format_confidence_val)
display(df_display)

print("\n🔬 Running round-trip metadata integrity assertions...")

# 1. Attribute existence and non-empty value assertions
assert "document_title" in live_aspect_data and bool(live_aspect_data["document_title"]), "Missing or empty document_title!"
assert "document_summary" in live_aspect_data and bool(live_aspect_data["document_summary"]), "Missing or empty document_summary!"
assert "extracted_entities" in live_aspect_data and bool(live_aspect_data["extracted_entities"]), "Missing or empty extracted_entities!"
assert "confidence_score" in live_aspect_data, "Missing mandatory confidence_score attribute!"

# 2. AI confidence operational threshold assertion (>= 0.90)
numeric_confidence = float(live_aspect_data.get("confidence_score", 0.0))
assert numeric_confidence >= 0.90, f"AI confidence ({numeric_confidence:.2f}) is below the 0.90 operational threshold!"

# 3. DataFrame non-empty row assertion
assert len(df_aspect) > 0, "DataFrame rendered empty without active rows!"

print("🎉 Round-trip metadata integrity verification PASSED: All structured attributes and confidence thresholds verified successfully!\n")


## Section 5: Verification, summary, and resource cleanup

### Summary and production event-driven deployment

In this cookbook, you have constructed an automated pipeline capable of unlocking and cataloging dark data across cloud object storage:

1. **AspectType engineering**: You programmatically registered `dark-data-extracted-metadata` with explicit protobuf field indices (`1..4`), ensuring strong typing for downstream AI and BI systems.
2. **Multimodal schema extraction**: You parsed sample product manuals from the `akanksha86/kc-retail-demo` demo repository using the global multimodal API endpoint (`gemini-3.6-flash`), extracting structured summaries and domain entities with high AI confidence.
3. **Catalog semantic binding**: You bound the extracted attributes to a custom Dataplex `Entry` using dot-separated map keys and verified complete **round-trip metadata integrity** via live backend inspection and explicit schema assertions (`EntryView.FULL`).

#### 🚀 Scaling to production: Event-driven automation
To deploy this architecture across cloud storage buckets in production:
- **Eventarc triggers**: Configure an **Eventarc** trigger bound to `google.cloud.storage.object.v1.finalized`. Whenever a new PDF agreement or technical manual is uploaded to GCS, Eventarc automatically invokes a **Cloud Run** service or serverless function running the Python extraction and binding logic demonstrated in this notebook.
- **AI agent grounding**: Because your unstructured documents are now structured, indexed, and cataloged inside **Knowledge Catalog**, downstream **BigQuery data agents** and **Model Context Protocol (MCP)** servers can query this exact `Aspect` metadata to ground enterprise LLM responses accurately.

---

### 5.1. Resource cleanup

In the code cell below, after verifying our round-trip assertions, we safely execute resource cleanup. This deletes the created catalog resources (`Entry`, `EntryGroup`, `EntryType`, `AspectType`) and temporary sample PDF files, leaving your Google Cloud project cleanly reset.


In [ ]:
# =====================================================================
# Section 5.1: Resource cleanup (Reset Google Cloud environment)
# =====================================================================
print("=======================================================")
print("🧹 Executing Section 5.1: Resource cleanup loop...")
print("=======================================================
")

entry_group_name = f"{parent_location}/entryGroups/{ENTRY_GROUP_ID}"
entry_name = f"{entry_group_name}/entries/{TARGET_DOCUMENT_ID}"
entry_type_name = f"{parent_location}/entryTypes/{ENTRY_TYPE_ID}"
aspect_type_name = f"{parent_location}/aspectTypes/{ASPECT_TYPE_ID}"

# 1. Delete Entry (`manual-lum-lig-des-8g8j`)
try:
    print(f"⌛ Deleting Entry: {entry_name} ...")
    catalog_client.delete_entry(name=entry_name)
    print("✅ Entry deleted successfully.")
except NotFound:
    print("ℹ️ Entry already deleted or not found.")

# 2. Delete EntryGroup (`retail-manuals-mesh`)
try:
    print(f"⌛ Deleting EntryGroup: {entry_group_name} ...")
    op = catalog_client.delete_entry_group(name=entry_group_name)
    if hasattr(op, "result"): op.result()
    print("✅ EntryGroup deleted successfully.")
except NotFound:
    print("ℹ️ EntryGroup already deleted or not found.")

# 3. Delete Custom EntryType (`retail-unstructured-doc`)
try:
    print(f"⌛ Deleting EntryType: {entry_type_name} ...")
    op = catalog_client.delete_entry_type(name=entry_type_name)
    if hasattr(op, "result"): op.result()
    print("✅ EntryType deleted successfully.")
except NotFound:
    print("ℹ️ EntryType already deleted or not found.")

# 4. Delete Custom AspectType (`dark-data-extracted-metadata`)
try:
    print(f"⌛ Deleting AspectType: {aspect_type_name} ...")
    op = catalog_client.delete_aspect_type(name=aspect_type_name)
    if hasattr(op, "result"): op.result()
    print("✅ AspectType deleted successfully.")
except NotFound:
    print("ℹ️ AspectType already deleted or not found.")

# 5. Remove local sample PDF asset from workspace
if os.path.exists("LUM-LIG-DES-8G8J_manual.pdf"):
    try:
        os.remove("LUM-LIG-DES-8G8J_manual.pdf")
        print("\n✅ Removed temporary local sample PDF file (`LUM-LIG-DES-8G8J_manual.pdf`).")
    except Exception as file_err:
        print(f"\nℹ️ Local file cleanup note: {file_err}")

print("\n✨ Clean up complete! Your Google Cloud environment and local workspace are cleanly reset.")
